In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Load the CSV files

In [5]:
btc = pd.read_csv("BTC_full_data.csv")
eth = pd.read_csv("ETH_full_data.csv")

btc.head()
eth.head()

,Date,Open,High,Low,Close,Adj Close,Volume,Log_Return
0,2018-01-01,755.757019,782.530029,742.004028,772.640991,772.640991,2595760128,NaN
1,2018-01-02,772.346008,914.830017,772.346008,884.443970,884.443970,5783349760,0.135145
2,2018-01-03,886.000000,974.471008,868.450989,962.719971,962.719971,5093159936,0.084803
3,2018-01-04,961.713013,1045.079956,946.085999,980.921997,980.921997,6502859776,0.018730
4,2018-01-05,975.750000,1075.390015,956.325012,997.719971,997.719971,6683149824,0.016980


# 2. evaluation function

In [1]:
def evaluate_strategy(strategy_returns, risk_free_rate=0.03, trading_days=365):
    r = strategy_returns.dropna()

    cumulative = np.exp(r.cumsum())

    cumulative_pnl = cumulative.iloc[-1] - 1
    avg_daily_pnl = r.mean()
    daily_vol = r.std() 
    annual_return = r.mean() * trading_days
    annual_vol = r.std() * np.sqrt(trading_days)

    sharpe = (annual_return - risk_free_rate) / annual_vol if annual_vol != 0 else np.nan

    peak = cumulative.cummax()
    drawdown = (cumulative - peak) / peak
    max_dd = drawdown.min()

    return {
        "Cumulative PnL": cumulative_pnl,
        "Average Daily PnL": avg_daily_pnl,
        "Daily Volatility": daily_vol, 
        "Annualised Return": annual_return,
        "Volatility": annual_vol,
        "Sharpe Ratio": sharpe,
        "Max Drawdown": max_dd
    }

# 3. Define Bollinger Band (price level)
position = 1 → long when price is below lower band
position = -1 → short when price is above upper band
position = 0 → exit when price reverts to the rolling mean

In [2]:
def bollinger_price_mean_reversion(
    df,
    date_col="Date",
    price_col="Close",
    return_col="Log_Return",
    window=20,
    num_std=2
):
    df = df.copy()

    # format date
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)

    # use price level for Bollinger signal
    df["price"] = df[price_col]

    # if Log_Return missing, calculate it
    if return_col not in df.columns:
        df[return_col] = np.log(df["price"] / df["price"].shift(1))

    # rolling stats on price level
    df["rolling_mean"] = df["price"].rolling(window).mean()
    df["rolling_std"] = df["price"].rolling(window).std()

    # bollinger bands
    df["upper_band"] = df["rolling_mean"] + num_std * df["rolling_std"]
    df["lower_band"] = df["rolling_mean"] - num_std * df["rolling_std"]

    # trading rule: symmetric mean reversion
    position = 0
    positions = []

    for i in range(len(df)):
        price = df.loc[i, "price"]
        mean = df.loc[i, "rolling_mean"]
        upper = df.loc[i, "upper_band"]
        lower = df.loc[i, "lower_band"]

        if pd.isna(mean) or pd.isna(upper) or pd.isna(lower):
            position = 0

        elif position == 0:
            if price < lower:
                position = 1      # long
            elif price > upper:
                position = -1     # short

        elif position == 1:
            if price >= mean:
                position = 0

        elif position == -1:
            if price <= mean:
                position = 0

        positions.append(position)

    df["position"] = positions

    # strategy log return using lagged position
    df["strategy_return"] = df["position"].shift(1).fillna(0) * df[return_col].fillna(0)

    return df
  

# 4. Run strategy and create final metrics table

In [6]:
btc_bb = bollinger_price_mean_reversion(btc)
eth_bb = bollinger_price_mean_reversion(eth)

btc_metrics = evaluate_strategy(btc_bb["strategy_return"])
eth_metrics = evaluate_strategy(eth_bb["strategy_return"])

btc_metrics, eth_metrics

({'Cumulative PnL': -0.7748770640708358,
  'Average Daily PnL': -0.0005104788237187843,
  'Daily Volatility': 0.02688110467633147,
  'Annualised Return': -0.18632477065735628,
  'Volatility': 0.5135627837433897,
  'Sharpe Ratio': -0.4212236118056533,
  'Max Drawdown': -0.9250128838783174},
 {'Cumulative PnL': -0.9494640893613103,
  'Average Daily PnL': -0.0010219346434979983,
  'Daily Volatility': 0.0345758934509815,
  'Annualised Return': -0.37300614487676936,
  'Volatility': 0.6605715168668516,
  'Sharpe Ratio': -0.6100870754892108,
  'Max Drawdown': -0.9701293923184195})

# 5. Result table

In [9]:
results = pd.DataFrame({
    "Criteria": [
        "Cumulative pnl",
        "Average daily pnl",
        "Annualised return",
        "Max drawdown",
        "Daily volatility", 
        "Volatility",
        "Sharpe Ratio (risk-free rate assumed to be 3%)"
    ],
    "BTC": [
        btc_metrics["Cumulative PnL"],
        btc_metrics["Average Daily PnL"],
        btc_metrics["Annualised Return"],
        btc_metrics["Max Drawdown"],
        btc_metrics["Daily Volatility"],
        btc_metrics["Volatility"],
        btc_metrics["Sharpe Ratio"]
    ],
    "ETH": [
        eth_metrics["Cumulative PnL"],
        eth_metrics["Average Daily PnL"],
        eth_metrics["Annualised Return"],
        eth_metrics["Max Drawdown"],
        eth_metrics["Daily Volatility"],
        eth_metrics["Volatility"],
        eth_metrics["Sharpe Ratio"]
    ]
})

results

,Criteria,BTC,ETH
0,Cumulative pnl,-0.774877,-0.949464
1,Average daily pnl,-0.000510,-0.001022
2,Annualised return,-0.186325,-0.373006
3,Max drawdown,-0.925013,-0.970129
4,Daily volatility,0.026881,0.034576
5,Volatility,0.513563,0.660572
6,Sharpe Ratio (risk-free rate assumed to be 3%),-0.421224,-0.610087


The Bollinger Bands mean-reversion strategy generated persistently negative performance for both BTC and ETH. Although short-term price reversals occasionally occurred, the strategy was unable to offset the impact of prolonged trending market regimes. This resulted in substantial cumulative losses, large maximum drawdowns exceeding 90%, and negative Sharpe ratios, indicating poor risk-adjusted returns. The weaker performance observed for ETH relative to BTC is consistent with its higher volatility and stronger directional price movements. Overall, the findings suggest that a simple Bollinger-based mean-reversion approach is not well suited to highly volatile and trend-dominated cryptocurrency markets without additional trend-filtering or risk-management mechanisms.